## DATA CLEANING

In [170]:
import pandas as pd

In [171]:
df=pd.read_csv('../data/marketing_campaign.csv')

In [172]:
df.head()

,ID\tYear_Birth\tEducation\tMarital_Status\tIncome\tKidhome\tTeenhome\tDt_Customer\tRecency\tMntWines\tMntFruits\tMntMeatProducts\tMntFishProducts\tMntSweetProducts\tMntGoldProds\tNumDealsPurchases\tNumWebPurchases\tNumCatalogPurchases\tNumStorePurchases\tNumWebVisitsMonth\tAcceptedCmp3\tAcceptedCmp4\tAcceptedCmp5\tAcceptedCmp1\tAcceptedCmp2\tComplain\tZ_CostContact\tZ_Revenue\tResponse
0,5524\t1957\tGraduation\tSingle\t58138\t0\t0\t0...
1,2174\t1954\tGraduation\tSingle\t46344\t1\t1\t0...
2,4141\t1965\tGraduation\tTogether\t71613\t0\t0\...
3,6182\t1984\tGraduation\tTogether\t26646\t1\t0\...
4,5324\t1981\tPhD\tMarried\t58293\t1\t0\t19-01-2...


In [173]:
df=pd.read_csv('../data/marketing_campaign.csv', sep="\t")

In [174]:
df.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0


In [175]:
df.shape

(2240, 29)

In [176]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
ID,2240.0,5592.159821,3246.662198,0.0,2828.25,5458.5,8427.75,11191.0
Year_Birth,2240.0,1968.805804,11.984069,1893.0,1959.00,1970.0,1977.00,1996.0
Income,2216.0,52247.251354,25173.076661,1730.0,35303.00,51381.5,68522.00,666666.0
Kidhome,2240.0,0.444196,0.538398,0.0,0.00,0.0,1.00,2.0
Teenhome,2240.0,0.506250,0.544538,0.0,0.00,0.0,1.00,2.0
Recency,2240.0,49.109375,28.962453,0.0,24.00,49.0,74.00,99.0
MntWines,2240.0,303.935714,336.597393,0.0,23.75,173.5,504.25,1493.0
MntFruits,2240.0,26.302232,39.773434,0.0,1.00,8.0,33.00,199.0
MntMeatProducts,2240.0,166.950000,225.715373,0.0,16.00,67.0,232.00,1725.0
MntFishProducts,2240.0,37.525446,54.628979,0.0,3.00,12.0,50.00,259.0


In [177]:
df.describe(include='object')

,Education,Marital_Status,Dt_Customer
count,2240,2240,2240
unique,5,8,663
top,Graduation,Married,31-08-2012
freq,1127,864,12


In [178]:
df.isnull().sum()

ID                      0
Year_Birth              0
Education               0
Marital_Status          0
Income                 24
Kidhome                 0
Teenhome                0
Dt_Customer             0
Recency                 0
MntWines                0
MntFruits               0
MntMeatProducts         0
MntFishProducts         0
MntSweetProducts        0
MntGoldProds            0
NumDealsPurchases       0
NumWebPurchases         0
NumCatalogPurchases     0
NumStorePurchases       0
NumWebVisitsMonth       0
AcceptedCmp3            0
AcceptedCmp4            0
AcceptedCmp5            0
AcceptedCmp1            0
AcceptedCmp2            0
Complain                0
Z_CostContact           0
Z_Revenue               0
Response                0
dtype: int64

In [179]:
df.duplicated().sum()

np.int64(0)

In [180]:
df['Education'].value_counts()

Education
Graduation    1127
PhD            486
Master         370
2n Cycle       203
Basic           54
Name: count, dtype: int64

In [181]:
df['Marital_Status'].value_counts()

Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64

## ⚠️ Issues Identified

- Missing values in the **Income** column.
- Outliers detected in the **Income** column.
- Incorrect data type for the **Dt_Customer** column.
- Verifying Unique Customer IDs
- Unrealistic **Year_Birth** column.
- Merging rare **Marital_Status** column.
- Removing **Z_CostContact** column and **Z_Revenue** column.

### 1. Verifying unique values in **ID** column.

In [182]:
df['ID'].nunique()

2240

### 2. Adding Missing Values for Income Column using Median of **Education** and **Marital_Status** columns.

In [183]:
df['Income'] = df['Income'].fillna(
    df.groupby('Education')['Income'].transform('median')
)

### 3. Finding and Removing Outliers from **Income** column.

In [184]:
Q1 = df['Income'].quantile(0.25)
Q3 = df['Income'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['Income'] < lower) | (df['Income'] > upper)]

print(outliers[['ID','Income']])

         ID    Income
164    8475  157243.0
617    1503  162397.0
655    5555  153924.0
687    1501  160803.0
1300   5336  157733.0
1653   4931  157146.0
2132  11181  156924.0
2233   9432  666666.0


In [185]:
df = df[
    (df['Income'] >= lower) &
    (df['Income'] <= upper)
]

In [186]:
df['Income'].describe()

count      2232.000000
mean      51635.562948
std       20603.164976
min        1730.000000
25%       35434.750000
50%       51381.500000
75%       68118.000000
max      113734.000000
Name: Income, dtype: float64

### 4. Changing data type of **Dt_Customer** column to date.

In [187]:
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%d-%m-%Y')

In [188]:
df['Dt_Customer']

0      2012-09-04
1      2014-03-08
2      2013-08-21
3      2014-02-10
4      2014-01-19
          ...    
2235   2013-06-13
2236   2014-06-10
2237   2014-01-25
2238   2014-01-24
2239   2012-10-15
Name: Dt_Customer, Length: 2232, dtype: datetime64[ns]

### 5. Removing unrealistic **Year_Birth** values.

In [189]:
current_year = pd.Timestamp.today().year

df['Age'] = current_year - df['Year_Birth']
df['Age'].describe()

count    2232.000000
mean       57.207437
std        11.990340
min        30.000000
25%        49.000000
50%        56.000000
75%        67.000000
max       133.000000
Name: Age, dtype: float64

In [190]:
df = df[(df['Age'] >= 18) & (df['Age'] <= 100)]
df['Age'].describe()

count    2229.000000
mean       57.111261
std        11.707430
min        30.000000
25%        49.000000
50%        56.000000
75%        67.000000
max        86.000000
Name: Age, dtype: float64

### 6. Merging rare values of **Marital_Status** column.

In [191]:
df['Marital_Status'] = df['Marital_Status'].replace({
    'Alone': 'Single',
    'YOLO': 'Single',
    'Absurd': 'Single'
})

In [192]:
df.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response,Age
0,5524,1957,Graduation,Single,58138.0,0,0,2012-09-04,58,635,...,0,0,0,0,0,0,3,11,1,69
1,2174,1954,Graduation,Single,46344.0,1,1,2014-03-08,38,11,...,0,0,0,0,0,0,3,11,0,72
2,4141,1965,Graduation,Together,71613.0,0,0,2013-08-21,26,426,...,0,0,0,0,0,0,3,11,0,61
3,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,...,0,0,0,0,0,0,3,11,0,42
4,5324,1981,PhD,Married,58293.0,1,0,2014-01-19,94,173,...,0,0,0,0,0,0,3,11,0,45


In [193]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
ID,2229.0,5589.296097,0.0,2829.0,5455.0,8420.0,11191.0,3244.840538
Year_Birth,2229.0,1968.888739,1940.0,1959.0,1970.0,1977.0,1996.0,11.70743
Income,2229.0,51624.146478,1730.0,35416.0,51373.0,68118.0,113734.0,20602.706289
Kidhome,2229.0,0.444594,0.0,0.0,0.0,1.0,2.0,0.538636
Teenhome,2229.0,0.507402,0.0,0.0,0.0,1.0,2.0,0.544735
Dt_Customer,2229,2013-07-10 05:20:25.841184512,2012-07-30 00:00:00,2013-01-16 00:00:00,2013-07-08 00:00:00,2013-12-30 00:00:00,2014-06-29 00:00:00,NaN
Recency,2229.0,49.106326,0.0,24.0,49.0,74.0,99.0,28.946476
MntWines,2229.0,304.991476,0.0,24.0,176.0,505.0,1493.0,336.761943
MntFruits,2229.0,26.348587,0.0,2.0,8.0,33.0,199.0,39.76406
MntMeatProducts,2229.0,165.283984,0.0,16.0,67.0,231.0,1725.0,219.336589


In [194]:
df = df.drop(columns=['Z_CostContact', 'Z_Revenue'])

In [195]:
column_order = [
    # Customer Information
    'ID',
    'Year_Birth',
    'Age',
    'Education',
    'Marital_Status',
    'Income',
    'Kidhome',
    'Teenhome',

    # Customer Details
    'Dt_Customer',
    'Recency',

    # Spending
    'MntWines',
    'MntFruits',
    'MntMeatProducts',
    'MntFishProducts',
    'MntSweetProducts',
    'MntGoldProds',

    # Purchase Behaviour
    'NumDealsPurchases',
    'NumWebPurchases',
    'NumCatalogPurchases',
    'NumStorePurchases',
    'NumWebVisitsMonth',

    # Marketing Campaigns
    'AcceptedCmp1',
    'AcceptedCmp2',
    'AcceptedCmp3',
    'AcceptedCmp4',
    'AcceptedCmp5',
    'Response',

    # Complaints
    'Complain'
]

df = df[column_order]

In [196]:
df.head()

,ID,Year_Birth,Age,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,...,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp1,AcceptedCmp2,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,Response,Complain
0,5524,1957,69,Graduation,Single,58138.0,0,0,2012-09-04,58,...,10,4,7,0,0,0,0,0,1,0
1,2174,1954,72,Graduation,Single,46344.0,1,1,2014-03-08,38,...,1,2,5,0,0,0,0,0,0,0
2,4141,1965,61,Graduation,Together,71613.0,0,0,2013-08-21,26,...,2,10,4,0,0,0,0,0,0,0
3,6182,1984,42,Graduation,Together,26646.0,1,0,2014-02-10,26,...,0,4,6,0,0,0,0,0,0,0
4,5324,1981,45,PhD,Married,58293.0,1,0,2014-01-19,94,...,3,6,5,0,0,0,0,0,0,0


In [197]:
df.reset_index(drop=True, inplace=True)

In [199]:
df.to_csv('../data/marketing_campaign_cleaned.csv',
    index=False
)